# 03 — Data Audit & Scientific Curation

**CuSelect-AI · Chapter III — From attributable evidence to scientifically defensible data**  
**Campaign:** `CU-001` · **Scientific focus:** Cu/Ag configurations for electrochemical CO₂ reduction to ethylene and ethanol  
**Input milestone:** Notebook 02 — validated evidence export (`809ec3c`, local Git milestone)  
**Notebook status:** Initial audit implemented; scientific curation and modeling approvals remain pending.

> **Guiding question:** Which literature-derived observations can be compared quantitatively, which are merely source-traceable evidence, and which require more experimental or computational context before modeling?

**Non-negotiable distinction:** A human-verified published observation is not automatically an ML training point; a numerically correct DFT value is not automatically transferable between Cu(111) and Cu(100) models.

### Baseline from Notebook 02

| Category | Starting count | Meaning |
|---|---:|---|
| Seed publications | 8 | Bibliographic starting set, **not** eight verified studies |
| Human-verified experimental claims | 3 | Source-checked observations from P005 |
| Computational descriptor candidates | 6 | Published source-cell-audited DFT values |
| Descriptors approved for modeling | 0 | No model compatibility established |
| Full RAG system ready | False | Retrieval/grounding has not been benchmarked |
| Blinded benchmark ready | False | No leakage-safe evaluation dataset |

**This notebook performs read-only audits.** It does not create human approvals, overwrite the review ledger, impute missing science, train an ML model, or export a purportedly curated training dataset.


## 3.1 — Research criteria and decision gates

A record can pass one quality gate but fail a different gate. We track these separately:

1. **Provenance integrity** — source publication, exact source location, unique evidence identifier, and unchanged approved fields.
2. **Measurement integrity** — observable, numerical value where applicable, units, reference scale, and limits.
3. **Experimental comparability** — catalyst identity, reactor, electrolyte, potential/current, iR treatment, product, and measurement basis.
4. **Computational comparability** — modeled facet, atomistic geometry, functional, reference states, definition of energy, and relevant electrochemical conditions.
5. **Evaluation validity** — sufficient independent publications and a publication-grouped split; no paper leakage.

**Decision vocabulary:** `source_verified`, `needs_manual_curation`, `not_model_approved`, and `not_benchmark_ready`. These are *audit labels*, not new approvals in the authoritative evidence register.


In [1]:
# PSEUDOCODE
# 1. Locate the CuSelect-AI root without assuming the notebook working directory.
# 2. Define the five authoritative Notebook 02 inputs plus the original workbook.
# 3. Stop with a clear error if any required file is missing.
# 4. Load text fields as strings so identifiers and missing-value markers are preserved.
# 5. Display an inventory; do not change any source file.

from pathlib import Path
import json
import math
import re
import pandas as pd
from IPython.display import display

start = Path.cwd().resolve()
ROOT = next((p for p in (start, *start.parents) if (p / "pyproject.toml").is_file()), None)
assert ROOT is not None, "Project root not found. Open VS Code in the CuSelect-AI project."

paths = {
    "papers": ROOT / "data/papers.csv",
    "human_reviews": ROOT / "data/references/02_human_reviewed_claims.csv",
    "verified_claims": ROOT / "data/processed/02_verified_claims.csv",
    "dft_candidates": ROOT / "data/interim/02_p005_computational_descriptor_candidates.csv",
    "readiness": ROOT / "outputs/reports/02_evidence_readiness.json",
    "source_workbook": ROOT / "data/raw/P005_source_data.xlsx",
}

required = ["papers", "human_reviews", "verified_claims", "dft_candidates", "readiness", "source_workbook"]
missing_files = [str(paths[name]) for name in required if not paths[name].is_file()]
assert not missing_files, "Missing input files (no data were created):\n" + "\n".join(missing_files)

papers = pd.read_csv(paths["papers"], dtype=str, keep_default_na=False)
reviews = pd.read_csv(paths["human_reviews"], dtype=str, keep_default_na=False)
claims = pd.read_csv(paths["verified_claims"], dtype=str, keep_default_na=False)
dft = pd.read_csv(paths["dft_candidates"], dtype=str, keep_default_na=False)
with paths["readiness"].open("r", encoding="utf-8") as handle:
    readiness = json.load(handle)

assert len(papers) == 8, "Unexpected number of seed publications. Reconcile Notebook 02 first."
assert len(claims) == 3, "Unexpected number of verified claims. Reconcile Notebook 02 first."
assert len(dft) == 6, "Unexpected DFT candidate count. Reconcile Notebook 02 first."
assert readiness.get("campaign_id") == "CU-001"

inventory = pd.DataFrame([
    {"input": "Publication register", "records": len(papers)},
    {"input": "Human-review register", "records": len(reviews)},
    {"input": "Verified experimental claims", "records": len(claims)},
    {"input": "Published DFT candidates", "records": len(dft)},
])
print("PROJECT:", ROOT)
print("CAMPAIGN:", readiness["campaign_id"])
display(inventory)
print("Input inventory: PASSED | No files modified")


PROJECT: /Users/nsolebna/202609_Ironhack/Week 8/Project 3/CuSelect-AI
CAMPAIGN: CU-001


,input,records
0,Publication register,8
1,Human-review register,3
2,Verified experimental claims,3
3,Published DFT candidates,6


Input inventory: PASSED | No files modified


## 3.2 — Confirm that evidence has not changed during handoff

The *authoritative* human-review CSV must agree **field for field** with the exported verified claims. Check complete rows, not only `evidence_id` or FE values. A source flag or review note changing silently is an integrity failure.

The audit deliberately does **not** assign `source_checked_by_human=True`, does not change `review_status`, and does not rewrite the export.


In [2]:
# PSEUDOCODE
# 1. Confirm required evidence identifiers and approval columns exist.
# 2. Detect duplicate record identifiers.
# 3. Select records approved by both human check and review status.
# 4. Compare every exported field to the approved original record.
# 5. Confirm all source publication IDs and DOIs are linked correctly.

required_review_columns = {"evidence_id", "paper_id", "doi", "review_status", "source_checked_by_human"}
required_paper_columns = {"paper_id", "doi"}
assert required_review_columns <= set(reviews.columns)
assert required_review_columns <= set(claims.columns)
assert required_paper_columns <= set(papers.columns)
assert reviews["evidence_id"].is_unique, "Duplicate human-reviewed evidence IDs"
assert claims["evidence_id"].is_unique, "Duplicate exported evidence IDs"
assert papers["paper_id"].is_unique, "Duplicate bibliography IDs"

human_checked = reviews["source_checked_by_human"].str.strip().str.lower().eq("true")
approved_status = reviews["review_status"].eq("human_verified")
assert not (approved_status & ~human_checked).any(), "Approved claim lacks an explicit human source check"

approved = reviews.loc[human_checked & approved_status].copy()
assert len(approved) == 3, "Expected three approved P005 experimental records"
assert set(approved.columns) == set(claims.columns), "Export schema differs from human-review schema"

column_order = sorted(approved.columns)
expected = approved[column_order].sort_values("evidence_id").reset_index(drop=True)
actual = claims[column_order].sort_values("evidence_id").reset_index(drop=True)
pd.testing.assert_frame_equal(expected, actual, check_dtype=False)

paper_dois = dict(zip(papers["paper_id"], papers["doi"].str.lower().str.strip()))
for row in claims.to_dict("records"):
    assert row["paper_id"] in paper_dois, f"Unknown source paper: {row['paper_id']}"
    assert row["doi"].lower().strip() == paper_dois[row["paper_id"]], (
        f"DOI-paper mismatch for {row['evidence_id']}"
    )
    assert row["review_status"] == "human_verified"
    assert row["source_checked_by_human"].strip().lower() == "true"

print("FULL-RECORD SOURCE SYNCHRONIZATION: PASSED")
print("Approved claims:", len(approved))
print("Fields checked per record:", len(column_order))
print("Publication and DOI linkage: PASSED")
print("No files modified")


FULL-RECORD SOURCE SYNCHRONIZATION: PASSED
Approved claims: 3
Fields checked per record: 17
Publication and DOI linkage: PASSED
No files modified


## 3.3 — Schema, field completeness, and missing-value vocabulary

Missing data are **scientific information**. An empty FE value in a product-ratio claim is different from an experimentally measured FE of zero. Likewise, the missing *CO reference in P005 Figure 5c must not become zero.

Here we report field-level missingness **without normalizing or overwriting the source**. A blank field does not always indicate a defect; some fields are conditional on the measurement type.


In [3]:
# PSEUDOCODE
# 1. Describe column names and representation for each source dataset.
# 2. Define missing-value markers for auditing only.
# 3. Count missing entries per field, while leaving the original strings untouched.
# 4. Display schema and missingness summaries.

MISSING_MARKERS = {"", "--", "not reported", "not available", "unknown", "null", "none", "nan"}

def missing_mask(series):
    values = series.fillna("").astype(str).str.strip().str.lower()
    return values.isin(MISSING_MARKERS)

schema_rows = []
missing_rows = []
for dataset_name, frame in [("papers", papers), ("human_reviews", reviews), ("verified_claims", claims), ("dft_candidates", dft)]:
    for column in frame.columns:
        n_missing = int(missing_mask(frame[column]).sum())
        schema_rows.append({"dataset": dataset_name, "column": column, "dtype": str(frame[column].dtype)})
        missing_rows.append({
            "dataset": dataset_name,
            "column": column,
            "missing_count": n_missing,
            "records": len(frame),
            "missing_fraction": round(n_missing / len(frame), 3) if len(frame) else None,
        })

schema_audit = pd.DataFrame(schema_rows)
missing_audit = pd.DataFrame(missing_rows)
print("COLUMN INVENTORY")
display(schema_audit)
print("MISSINGNESS (includes conditionally inapplicable fields)")
display(missing_audit.sort_values(["dataset", "missing_count", "column"], ascending=[True, False, True]))
print("Original strings retained; no imputation performed")


COLUMN INVENTORY


,dataset,column,dtype
0,papers,paper_id,object
1,papers,year,object
2,papers,title,object
3,papers,doi,object
4,papers,kind,object
5,papers,why_relevant,object
6,human_reviews,evidence_id,object
7,human_reviews,paper_id,object
8,human_reviews,doi,object
9,human_reviews,claim_paraphrase,object


MISSINGNESS (includes conditionally inapplicable fields)


,dataset,column,missing_count,records,missing_fraction
40,dft_candidates,catalyst,0,6,0.000
50,dft_candidates,descriptor_approved,0,6,0.000
47,dft_candidates,doi,0,6,0.000
48,dft_candidates,evidence_origin,0,6,0.000
49,dft_candidates,model_compatibility,0,6,0.000
41,dft_candidates,quantity,0,6,0.000
46,dft_candidates,reference_note,0,6,0.000
45,dft_candidates,source_cell,0,6,0.000
44,dft_candidates,source_sheet,0,6,0.000
43,dft_candidates,unit,0,6,0.000


Original strings retained; no imputation performed


## 3.4 — Identify measurement types before numerical analysis

P005 contains **one ethanol-to-ethylene product-ratio comparison** and **two product-specific maximum Faradaic-efficiency observations**. These are *different observables*. They cannot be pooled into a numerical target column simply because they involve C₂ products.

The ratio claim remains an attributed text observation; the structured `faradaic_efficiency_percent` values apply only where actually provided by the approved records.


In [4]:
# PSEUDOCODE
# 1. Select existing scientific-type and target columns from verified evidence.
# 2. Parse only the dedicated Faradaic-efficiency field into numeric form.
# 3. Check that any FE is finite and physically within 0–100 percent.
# 4. Flag missing scientific context for subsequent manual curation.
# 5. Do not parse the product-ratio text as a Faradaic efficiency.

measurement_columns = [
    "evidence_id", "paper_id", "claim_type", "catalyst_system",
    "target_or_observation", "faradaic_efficiency_percent", "fe_product",
    "conditions", "source_extent", "exact_source_location"
]
assert set(measurement_columns) <= set(claims.columns), "Evidence schema changed; inspect required columns"
measurements = claims[measurement_columns].copy()

fe_text = measurements["faradaic_efficiency_percent"].str.strip()
fe_present = ~missing_mask(fe_text)
fe_numeric = pd.to_numeric(fe_text.where(fe_present), errors="coerce")
assert fe_numeric.loc[fe_present].notna().all(), "A reported FE is not numeric"
assert ((fe_numeric.loc[fe_present] >= 0) & (fe_numeric.loc[fe_present] <= 100)).all(), (
    "FE outside the physically permissible 0–100% interval"
)

measurements["FE_percent_for_audit"] = fe_numeric
measurements["FE_reported"] = fe_present
measurements["manual_comparability_review"] = True

display(measurements[[
    "evidence_id", "claim_type", "catalyst_system", "target_or_observation",
    "fe_product", "FE_percent_for_audit", "FE_reported"
]])
print("Records with a structured FE:", int(fe_present.sum()))
print("Records without a structured FE (not zero):", int((~fe_present).sum()))
print("No pooled target, normalization, or statistical independence assumed")


,evidence_id,claim_type,catalyst_system,target_or_observation,fe_product,FE_percent_for_audit,FE_reported
0,E-DRAFT-001,experimental_observation,AgCu NW and Ag1Cu NW,Ethanol-to-ethylene product ratio,,NaN,False
1,E-P005-FE-C2H4-001,experimental_observation,AgCu NW,Ethylene FE,C2H4,54.9,True
2,E-P005-FE-ETOH-001,experimental_observation,Ag1Cu NW,Ethanol FE,C2H5OH,56.3,True


Records with a structured FE: 2
Records without a structured FE (not zero): 1
No pooled target, normalization, or statistical independence assumed


## 3.5 — Experimental-context completeness and comparability

A defensible comparison requires aligned **reactor configuration**, **electrolyte**, **potential scale and iR treatment**, **product**, and **measurement basis**. These fields are currently embedded in human-reviewed text. We display them verbatim instead of silently guessing a numerical potential or electrolyte concentration.

**Known P005 context:** the reported maximum FE observations were obtained using a flow cell and 1 M KOH, at different applied potentials (−1.1 and −1.0 V vs RHE), without iR correction. They therefore do not form a controlled same-potential catalyst comparison.

**Future controlled curation:** introduce separate manually reviewed numeric fields for potential, electrolyte composition, temperature, current density, gas feed, and reactor geometry. Preserve the unedited source text and per-field provenance.


In [5]:
# PSEUDOCODE
# 1. Display the original experimental contexts and measurement identifiers.
# 2. Flag absent core context fields for later manual review.
# 3. Build a *read-only* audit view rather than an approved curated dataset.
# 4. Explicitly keep comparative modeling permission false.

context_fields = [
    "evidence_id", "paper_id", "catalyst_system", "target_or_observation",
    "conditions", "exact_source_location", "attribution_note", "split_group"
]
assert set(context_fields) <= set(claims.columns)
context_audit = claims[context_fields].copy()
for field in ["catalyst_system", "target_or_observation", "conditions", "exact_source_location", "split_group"]:
    context_audit[f"{field}_present"] = ~missing_mask(context_audit[field])
context_audit["structured_conditions_reviewed"] = False
context_audit["comparable_for_quantitative_ML"] = False

print("ORIGINAL SOURCE ATTRIBUTION AND CONDITIONS")
display(context_audit[[
    "evidence_id", "catalyst_system", "target_or_observation", "conditions",
    "exact_source_location", "structured_conditions_reviewed",
    "comparable_for_quantitative_ML"
]])
print("Structured condition reviews completed: 0 (not performed here)")
print("No new scientific claims or dataset writes")


ORIGINAL SOURCE ATTRIBUTION AND CONDITIONS


,evidence_id,catalyst_system,target_or_observation,conditions,exact_source_location,structured_conditions_reviewed,comparable_for_quantitative_ML
0,E-DRAFT-001,AgCu NW and Ag1Cu NW,Ethanol-to-ethylene product ratio,,"Wang et al. (2024), Abstract",False,False
1,E-P005-FE-C2H4-001,AgCu NW,Ethylene FE,Flow cell; 1 M KOH; -1.1 V vs RHE; no iR corre...,CO2RR performance; Fig. 2c; Supplementary Figs...,False,False
2,E-P005-FE-ETOH-001,Ag1Cu NW,Ethanol FE,Flow cell; 1 M KOH; -1.0 V vs RHE; no iR corre...,CO2RR performance; Fig. 2c; Supplementary Figs...,False,False


Structured condition reviews completed: 0 (not performed here)
No new scientific claims or dataset writes


## 3.6 — Computational descriptors: original-source reproducibility

The six candidate descriptors originate from **published P005 DFT**, not from any computation by CuSelect-AI. The source spreadsheet uses Cu(111)-based models, whereas the project's exploratory ASE structures are idealized Cu(100) slabs.

This audit distinguishes:

- **H₂O adsorption energies** from **reported *CHO free-energy levels**.
- Numeric source-cell fidelity from model transferability.
- A tabulated *CO reference from an absent reference (never impute zero).
- Source-audited numeric values from modeling-approved descriptors.


In [6]:
# PSEUDOCODE
# 1. Check the six descriptor rows and their provenance fields.
# 2. Verify the model-approval flags remain false.
# 3. Confirm quantity, unit, source DOI, and source-cell identity.
# 4. Read the original publisher Excel workbook in read-only mode.
# 5. Compare each saved numerical value against its original cell.
# 6. Leave all source files unchanged.

from openpyxl import load_workbook

required_dft_columns = {
    "catalyst", "quantity", "value", "unit", "source_sheet", "source_cell",
    "doi", "evidence_origin", "model_compatibility", "descriptor_approved"
}
assert required_dft_columns <= set(dft.columns)
assert dft["source_sheet"].ne("").all() and dft["source_cell"].ne("").all()
assert dft["unit"].eq("eV").all(), "Unexpected energy unit"
assert dft["evidence_origin"].eq("published_DFT").all()
assert dft["doi"].str.lower().eq("10.1038/s41467-024-54636-w").all()
assert dft["model_compatibility"].eq("not_established").all()
assert dft["descriptor_approved"].str.strip().str.lower().eq("false").all()
assert set(dft["quantity"]) == {"H2O_adsorption_energy", "CHO_free_energy_level"}

source_workbook = load_workbook(paths["source_workbook"], read_only=True, data_only=True)
source_matches = []
try:
    for record in dft.to_dict("records"):
        sheet = record["source_sheet"]
        cell = record["source_cell"]
        assert sheet in source_workbook.sheetnames, f"Missing original worksheet {sheet}"
        source_value = source_workbook[sheet][cell].value
        assert source_value is not None, f"Original cell {sheet}!{cell} is blank"
        stored_value = float(record["value"])
        exact_match = math.isclose(stored_value, float(source_value), rel_tol=0, abs_tol=1e-6)
        source_matches.append({
            "quantity": record["quantity"], "catalyst": record["catalyst"],
            "source": f"{sheet}!{cell}", "stored_eV": stored_value,
            "original_eV": float(source_value), "matched": exact_match,
        })
finally:
    source_workbook.close()

source_audit = pd.DataFrame(source_matches)
assert len(source_audit) == 6 and source_audit["matched"].all(), "DFT/source mismatch"
print("PUBLISHED DFT — NUMERICAL SOURCE AUDIT")
display(source_audit)
print("Cell-level source matches:", int(source_audit["matched"].sum()), "/", len(source_audit))
print("DFT descriptors approved for modeling: 0")
print("Source workbook not modified")


PUBLISHED DFT — NUMERICAL SOURCE AUDIT


,quantity,catalyst,source,stored_eV,original_eV,matched
0,H2O_adsorption_energy,Cu NW,Figure 5a!D5,-0.06094,-0.06094,True
1,H2O_adsorption_energy,Ag1Cu NW,Figure 5a!D6,-0.70452,-0.70452,True
2,H2O_adsorption_energy,AgCu NW,Figure 5a!D7,-0.04679,-0.04679,True
3,CHO_free_energy_level,CuNW,Figure 5c!D7,1.16000,1.16000,True
4,CHO_free_energy_level,Ag1Cu NW,Figure 5c!E7,0.95000,0.95000,True
5,CHO_free_energy_level,HO-Ag1Cu NW,Figure 5c!F7,0.83000,0.83000,True


Cell-level source matches: 6 / 6
DFT descriptors approved for modeling: 0
Source workbook not modified


## 3.7 — Applicability domain and grouping for future model evaluation

A source's observations must not be split randomly between training and testing: two claims from the same paper are **not** independent blind test cases. Use **publication-level grouping** (`doi` or `split_group`) when a sufficiently broad validated dataset exists.

For this starting dataset, all three verified experimental records originate in **one publication**. A meaningful publication-held-out performance estimate is **not possible**. Do not create a train/test split or report cross-validation metrics.


In [7]:
# PSEUDOCODE
# 1. Count independent publications represented in approved claims.
# 2. Verify all split_group values agree with the source DOI.
# 3. Calculate which seed papers lack human-verified evidence.
# 4. Reject any attempt to declare a valid held-out benchmark from one source.

assert "split_group" in claims.columns
assert claims["split_group"].str.lower().str.strip().eq(claims["doi"].str.lower().str.strip()).all(), (
    "Source split-group mismatch; investigate before benchmarking"
)
coverage = (
    claims.groupby(["paper_id", "doi", "split_group"], dropna=False)
    .agg(verified_claims=("evidence_id", "count"))
    .reset_index()
)
source_count = int(claims["doi"].nunique())
missing_papers = sorted(set(papers["paper_id"]) - set(claims["paper_id"]))
benchmark_possible = source_count >= 2  # Necessary, NOT sufficient for credible ML evaluation.
assert source_count == 1, "Evidence baseline has changed; reassess the benchmark protocol"
assert not benchmark_possible

print("PUBLICATION-LEVEL COVERAGE")
display(coverage)
print("Independent publications with verified evidence:", source_count)
print("Seed papers lacking approved evidence:", ", ".join(missing_papers))
print("Leakage-safe benchmark possible:", benchmark_possible)
print("No train/test split created")


PUBLICATION-LEVEL COVERAGE


,paper_id,doi,split_group,verified_claims
0,P005,10.1038/s41467-024-54636-w,10.1038/s41467-024-54636-w,3


Independent publications with verified evidence: 1
Seed papers lacking approved evidence: P001, P002, P003, P004, P006, P007, P008
Leakage-safe benchmark possible: False
No train/test split created


## 3.8 — Cross-check the evidence-readiness report

The readiness JSON is a **status artifact**, not a substitute for reading or evaluating sources. Its counts must agree with the actual CSVs. `verified_claims_eligible_for_retrieval` does not mean that a complete, benchmarked RAG system exists.

This audit never changes the report; it stops if counts or claimed readiness differ from the evidence.


In [8]:
# PSEUDOCODE
# 1. Recompute evidence counts using the original inputs.
# 2. Compare these values to the stored Notebook 02 readiness report.
# 3. Confirm RAG, evaluation and prediction readiness are not overstated.
# 4. Display an accurate read-only status summary.

expected_counts = {
    "seed_papers": len(papers),
    "human_verified_claims": len(approved),
    "publications_with_verified_claims": int(claims["paper_id"].nunique()),
    "publications_without_verified_claims": len(papers) - int(claims["paper_id"].nunique()),
    "verified_claims_eligible_for_retrieval": len(claims),
    "computational_descriptor_candidates": len(dft),
    "approved_modeling_descriptors": 0,
}
for label, expected_value in expected_counts.items():
    assert readiness.get(label) == expected_value, (
        f"Outdated Notebook 02 readiness field {label}: "
        f"expected {expected_value}, got {readiness.get(label)}"
    )
for label in ["evidence_ready_for_rag", "ready_for_blinded_benchmark", "scientific_selectivity_predictions_available"]:
    assert readiness.get(label) is False, f"Unsupported scientific readiness flag: {label}"

readiness_audit = pd.DataFrame([
    {"metric": label, "recomputed": value, "stored": readiness.get(label), "matched": value == readiness.get(label)}
    for label, value in expected_counts.items()
])
display(readiness_audit)
print("All readiness counts and conservative status flags: PASSED")
print("No report modified")


,metric,recomputed,stored,matched
0,seed_papers,8,8,True
1,human_verified_claims,3,3,True
2,publications_with_verified_claims,1,1,True
3,publications_without_verified_claims,7,7,True
4,verified_claims_eligible_for_retrieval,3,3,True
5,computational_descriptor_candidates,6,6,True
6,approved_modeling_descriptors,0,0,True


All readiness counts and conservative status flags: PASSED
No report modified


## 3.9 — Scientific interpretation and open curation tasks

**What the initial audits establish:**

- Three full records retain their human-verification provenance and exactly match the Notebook 02 export.
- Two FE measurements can be checked numerically, but the product-ratio claim is **not** an FE measurement.
- Six published DFT values can be traced to exact source-workbook cells.
- No DFT descriptor is yet comparable to the project's exploratory Cu(100) model without a justified mapping or new calculations.
- One source publication is insufficient for publication-held-out ML evaluation.

**Not yet established:** measurement-level independence, uncertainty intervals consistently encoded across publications, comparable testing protocols, quantitative causal Ag-site effects, predictive selectivity, or benchmarked RAG quality.

### Next manual scientific curation queue

1. Define normalized schema fields and preserve raw values in parallel.
2. Verify catalyst structure and Ag-site definitions; do not conflate nanowires with idealized slabs.
3. Collect experimental conditions with provenance for each individual observation.
4. Distinguish maxima across potentials from matched-condition comparisons.
5. Verify replicate count and error-bar meaning from each paper or supplementary source.
6. Expand verification to additional papers before model training or benchmarking.
7. Agree on train/test grouping, missingness strategy, and applicability domain **before** training.

**No curation output is automatically approved in this notebook.**


In [9]:
# PSEUDOCODE
# 1. Assemble an in-memory summary of the audited evidence base.
# 2. Keep model-readiness and benchmark-readiness false.
# 3. Report the next scientific work without writing any files.

notebook03_summary = {
    "campaign_id": "CU-001",
    "source_notebook": "02_Literature_and_Evidence",
    "seed_publications": len(papers),
    "source_verified_experimental_records": len(approved),
    "independent_publications_with_verified_claims": source_count,
    "published_DFT_candidates": len(dft),
    "source_cell_matches": int(source_audit["matched"].sum()),
    "model_approved_descriptors": 0,
    "quantitative_ML_dataset_approved": False,
    "publication_held_out_benchmark_ready": False,
    "full_RAG_validated": False,
    "human_review_register_modified": False,
    "curated_dataset_written": False,
}
print("NOTEBOOK 03 — INITIAL SCIENTIFIC AUDIT")
print(json.dumps(notebook03_summary, indent=2, ensure_ascii=False))
print("\nINITIAL READ-ONLY AUDIT COMPLETE. Further human curation required.")


NOTEBOOK 03 — INITIAL SCIENTIFIC AUDIT
{
  "campaign_id": "CU-001",
  "source_notebook": "02_Literature_and_Evidence",
  "seed_publications": 8,
  "source_verified_experimental_records": 3,
  "independent_publications_with_verified_claims": 1,
  "published_DFT_candidates": 6,
  "source_cell_matches": 6,
  "model_approved_descriptors": 0,
  "quantitative_ML_dataset_approved": false,
  "publication_held_out_benchmark_ready": false,
  "full_RAG_validated": false,
  "human_review_register_modified": false,
  "curated_dataset_written": false
}

INITIAL READ-ONLY AUDIT COMPLETE. Further human curation required.


## 3.10 — Handoff for the next guided session

**Completed in this notebook version:** A runnable, read-only evidence handoff audit; schema and missingness inspection; FE type checks; condition completeness inspection; publisher-cell verification of DFT values; source publication coverage; and conservative readiness reconciliation.

**Next guided step:** Develop a *reviewable, versioned curation schema* separating raw and structured fields. Begin with one P005 FE observation before expanding to other publications. Do not mass-extract conditions via regex without confirmation, and do not add any automatic `human_verified` or `descriptor_approved` assignments.

**Operational note:** This notebook expects the existing CuSelect-AI project with its local Notebook 02 outputs and `data/raw/P005_source_data.xlsx`. The raw workbook is deliberately treated as an external primary source rather than embedded in the notebook. If it is absent, stop and restore the actual publisher file rather than creating example data.

**Execution method:** Open inside the existing project, select the `cuselect` kernel, and choose **Restart → Run All**. The notebook deliberately performs no file writes. After review, commit the notebook in Git; generated output folders need not be included.
